# 00 · Quickstart — prove the pipeline works (3 minutes, no data needed)


> **Research prototype — not a medical device.** Nothing produced by these notebooks may be
> used to diagnose, treat, or make any decision about a patient.


This notebook runs **every stage** — preprocessing, diffusion, synthesis, the
vision-language backbone, the hypergraph, evaluation and export — on
procedurally generated films. No PhysioNet account, no downloads, no GPU.

Run it first. If it finishes, your environment is correct and any later problem
is about the data, not the code.

In [ ]:
import subprocess, sys
print(sys.version)
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip() or "no GPU reported")
except FileNotFoundError:
    print("nvidia-smi not found - you are on CPU. Runtime > Change runtime type > T4 GPU.")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# --- 1. where results live -------------------------------------------------
# Mounting Drive is strongly recommended: Colab disconnects, and every stage
# here writes a resumable checkpoint. Without Drive you start over.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = '/content/drive/MyDrive/dvlhg'
else:
    RUN_ROOT = '/content/dvlhg'

# --- 2. get the code -------------------------------------------------------
# Pick ONE. 'clone' is easiest once you have pushed this repo to GitHub.
SOURCE = 'clone'        # 'clone' | 'zip' | 'drive'
REPO_URL = 'https://github.com/abelsangeeth/DVL-Hyperparameter-for-Lung-Disease-Diagnosis.git'
ZIP_PATH = '/content/dvl-hypergraph.zip'          # if SOURCE == 'zip'
DRIVE_CODE = '/content/drive/MyDrive/dvl-hypergraph'  # if SOURCE == 'drive'

import os, shutil, subprocess, sys
CODE = '/content/dvl-hypergraph'
if not os.path.exists(CODE):
    if SOURCE == 'clone':
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, CODE], check=True)
    elif SOURCE == 'zip':
        if not os.path.exists(ZIP_PATH):
            from google.colab import files
            up = files.upload()            # choose the zip from scripts/make_colab_zip.py
            ZIP_PATH = '/content/' + next(iter(up))
        shutil.unpack_archive(ZIP_PATH, '/content/')
    elif SOURCE == 'drive':
        shutil.copytree(DRIVE_CODE, CODE)
print('code at', CODE, '| contents:', sorted(os.listdir(CODE))[:8])

# --- 3. dependencies -------------------------------------------------------
# Colab already ships torch/torchvision built for its CUDA - never reinstall them.
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'open_clip_torch>=2.24', 'timm>=0.9.12', 'transformers>=4.35',
                'fastapi', 'uvicorn', 'python-multipart'], check=True)

sys.path.insert(0, os.path.join(CODE, 'src'))
os.chdir(CODE)
os.environ['PYTHONPATH'] = os.path.join(CODE, 'src')
os.environ['RUN_ROOT'] = RUN_ROOT
print('run root ->', RUN_ROOT)

### The smoke run

`dvlhg smoke` builds ~900 synthetic chest films with real, label-correlated
structure (an enlarged cardiac silhouette for Cardiomegaly, a blunted
costophrenic angle for Pleural Effusion, a basal band for Atelectasis,
perihilar haze for Edema), then runs the whole pipeline on them.

The four ablation numbers it prints at the end should be **above chance**. They
are not meaningful clinical numbers — the point is that every component
connects and learns.

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'dvlhg.cli', 'smoke',
                '--n', '900', '--epochs', '8',
                '--root', os.path.join(os.environ['RUN_ROOT'], 'smoke')], check=True)

### What the synthetic films look like

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from dvlhg.data.synthetic import render_cxr
from dvlhg.constants import LABELS

rng = np.random.default_rng(0)
cases = [np.zeros(4, np.float32)] + [np.eye(4, dtype=np.float32)[i] for i in range(4)]
names = ['no finding'] + LABELS
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
for ax, vector, name in zip(axes, cases, names):
    ax.imshow(render_cxr(vector, 256, rng), cmap='gray', vmin=0, vmax=1)
    ax.set_title(name, fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

### Next

- **01** prepares the real dataset (MIMIC-CXR-JPG, or Open-i if you have no PhysioNet access)
- **02** trains the diffusion model
- **03** trains the vision-language backbone
- **04** builds the hypergraph, evaluates, and exports the serving bundle
- **05** runs the web demo